# R reproduction of kaggle Interactive Maps course

# Introduction

In this tutorial, you'll learn how to create interactive maps with the leaflet package. Along the way, you'll apply your new skills to visualize Boston crime data.

## leaflet docs
<https://rstudio.github.io/leaflet/>


In [ ]:
library(here)
library(tidyverse)
library(sf)
library(leaflet)
library(leaflet.extras)
library(units)

# Your first interactive map
We begin by creating a relatively simple map

In [ ]:
# create a map using leaflet's default open street map tiles
m_1 <- leaflet() %>% 
    addTiles() %>% 
    setView(lng = -71.0589, lat = 42.32, zoom = 12)

# display the map
m_1

Several arguments customize the appearance of the map:

* `setView` sets the initial center of the map. We use the latitude (42.32° N) and longitude (-71.0589° E) of the city of Boston. (Note: use order of lng, lat)
* `addTiles` changes the styling of the map; in this case, we choose the [OpenStreetMap](https://www.openstreetmap.org/#map=10/42.32/-71.0589) style. If you're curious, you can find the other options listed [here](https://rstudio.github.io/leaflet/reference/providers.html).
* `zoom` sets the initial level of zoom of the map, where higher values zoom in closer to the map.
Take the time now to explore by zooming in and out, or by dragging the map in different directions.

# The data
Now, we'll add some crime data to the map!

In [ ]:
crimes <- read_csv(here("data/crimes-in-boston/crimes-in-boston/crime.csv"))

In [ ]:
# Drop rows with missing locations
# Focus on major crimes since 2018

crimes <- crimes %>% 
    drop_na(c('Lat', 'Long', 'DISTRICT')) %>%
    filter(OFFENSE_CODE_GROUP %in% c(
        'Larceny', 
        'Auto Theft', 
        'Robbery', 
        'Larceny From Motor Vehicle', 
        'Residential Burglary', 
        'Simple Assault', 
        'Harassment', 
        'Ballistics', 
        'Aggravated Assault', 
        'Other Burglary', 
        'Arson', 
        'Commercial Burglary', 
        'HOME INVASION', 
        'Homicide', 
        'Criminal Harassment', 
        'Manslaughter'
        )
    ) %>% 
    filter(YEAR >= 2018)

In [ ]:
head(crimes)

## Plotting points
To reduce the amount of data we need to fit on the map, we'll (temporarily) confine our attention to daytime robberies.

In [ ]:
daytime_robberies <- crimes %>% 
    filter(OFFENSE_CODE_GROUP == "Robbery") %>% 
    filter(HOUR %in% 9:17)

In [ ]:
head(daytime_robberies)

Marker

In [ ]:
leaflet(daytime_robberies) %>% 
    setView(lng = -71.0589, lat = 42.32, zoom = 12) %>% 
    addProviderTiles("CartoDB.Positron") %>% 
    addMarkers(lng = ~Long, 
               lat = ~Lat,
               clusterOptions = markerClusterOptions()
    )

qq
## bubble map

In [ ]:
color_producer <- function(x) {
    # ifelse(x <= 12, 'forestgreen', "darkred") # most concise for this use
    color_vec <- c()
    for (i in seq_along(x)) {
        if (x[i] <= 12) {
            color_vec <- c(color_vec, "forestgreen")
        } else {
            color_vec <- c(color_vec, "darkred")
        }
    }
    return(color_vec)
}

leaflet(daytime_robberies) %>% 
    setView(lng = -71.0589, lat = 42.32, zoom = 12) %>% 
    addProviderTiles("CartoDB.Positron") %>%
    addCircles(lng = ~Long, 
               lat = ~Lat,
               radius = 20,
               color = ~color_producer(HOUR)
    )

## Heatmap

In [ ]:
leaflet(crimes) %>% 
    setView(lng = -71.0589, lat = 42.32, zoom = 12) %>% 
    addProviderTiles("CartoDB.Positron") %>%
    addHeatmap(lng = ~Long, 
               lat = ~Lat,
               radius = 10,
               blur = 15, 
               max = 1,
               minOpacity = 0.5
               )


## Choropleth map

In [ ]:
districts_full <- read_sf(here("data/Police_Districts/Police_Districts/Police_Districts.shp"))

In [ ]:
head(districts_full)

In [ ]:
districts <- districts_full %>% 
    select(c(DISTRICT, geometry))

In [ ]:
head(districts)

In [ ]:
plot_df <- crimes %>% 
    count(DISTRICT, sort = FALSE)
    # plot_df$n <- as.factor(plot_df$n)

In [ ]:
head(plot_df)

In [ ]:
# joined df not actually needed
districts_joined <- districts %>% 
    left_join(plot_df, by = c("DISTRICT" = "DISTRICT"))

### setup a color pallete and labels for popups

In [ ]:
pal <- colorNumeric("YlGnBu", plot_df$n)
labels <- sprintf(
  "<strong>BPD district %s</strong><br/>%d crimes",
  districts$DISTRICT, plot_df$n
) %>% lapply(htmltools::HTML)

In [ ]:
leaflet(districts_joined) %>% 
    setView(lng = -71.0589, lat = 42.32, zoom = 12) %>% 
    addProviderTiles("CartoDB.Positron") %>% 
    addPolygons(color = "black",
                weight = 0.5,
                opacity = 0.8,
                fillColor = ~pal(plot_df$n), 
                fillOpacity = 0.7,
                highlightOptions = highlightOptions(
                    weight = 1,
                    opacity = 1,
                    fillOpacity = 0.7,
                    bringToFront = TRUE
                ),
                label = labels,
                labelOptions = labelOptions(
                    style = list("font-weight" = "normal", padding = "3px 8px"),
                    textsize = "15px",
                    direction = "auto"
                )
    ) %>% 
    addLegend(pal = pal, 
              values = ~plot_df$n, 
              opacity = 0.7, 
              title = NULL,
                position = "bottomright"
    )


# Exercises

In [ ]:
plate_boundaries <- read_sf(here("data/Plate_Boundaries/Plate_Boundaries/Plate_Boundaries.shp"))

In [ ]:
head(plate_boundaries)

### Setting up the polylines

In [ ]:
# don't think I need this to create the polylines, but wanted to see how it was done
# to match the Python example
coords_df <- plate_boundaries %>% 
    mutate(coordinates = map(geometry, ~ { 
        coords <- st_coordinates(.)
        map(seq_len(nrow(coords)), function(i) c(coords[i, "X"], coords[i, "Y"]))
        }))

leaflet is great, you can just pass a sfc MULTILINESTRING object to addPolylines

First combine the LINESTRINGs into a MULTILINESTRING 

In [ ]:
pb_multiline <- st_combine(plate_boundaries$geometry)

In [ ]:
leaflet() %>% 
    setView(lng = 136, lat = 35, zoom = 5) %>% 
    addProviderTiles("CartoDB.Positron") %>% 
    addPolylines(data = pb_multiline, color = "black")


now we add earthquakes

In [ ]:
earthquakes <- read_csv(here("data/earthquakes1970-2014.csv"))

In [ ]:
leaflet() %>% 
    setView(lng = 136, lat = 35, zoom = 5) %>% 
    addProviderTiles("CartoDB.Positron") %>% 
    addPolylines(data = pb_multiline, color = "black") %>% 
    addHeatmap(data = earthquakes,
               lng = ~Longitude, 
               lat = ~Latitude, 
               radius = 10, 
               minOpacity = 0.5)

In [ ]:
color_producer <- function(x) {
    color_vec <- c()
    for (i in seq_along(x)) {
        if (x[i] < 50) {
            color_vec <- c(color_vec, "blue")
        } else if (x[i] < 100) {
            color_vec <- c(color_vec, "green")
        } else {
            color_vec <- c(color_vec, "red")
        }
    }
    return(color_vec)
}

leaflet() %>% 
    setView(lng = 136, lat = 35, zoom = 5) %>% 
    addProviderTiles("CartoDB.Positron") %>% 
    addPolylines(data = pb_multiline, color = "black") %>%
    addCircles(data = earthquakes,
               lng = ~Longitude, 
               lat = ~Latitude,
               radius = 2000,
               color = ~color_producer(earthquakes$Depth)
               )

Japan's prefectures

In [ ]:
prefectures = read_sf(here("data/japan-prefecture-boundaries/japan-prefecture-boundaries/japan-prefecture-boundaries.shp"))

population stats

In [ ]:
population <- read_csv(here("data/japan-prefecture-population.csv"))

In [ ]:
stats <- tibble(prefecture = prefectures$prefecture,
       area_sqkm = 
           st_transform(st_geometry(prefectures), 32654) %>%
           st_area() %>%
           set_units(km^2) %>% 
           as.double()
       ) %>%
    left_join(population, by = "prefecture") %>% 
    mutate(density = population / area_sqkm)

In [ ]:
stats

Palettes, popups, and labels

In [ ]:
pal <- colorNumeric("BuPu", stats$density)

labels <- sprintf(
    "<strong>%s Prefecture</strong><br/>Pop. Density: %.2f ",
    stats$prefecture, stats$density) %>%
    lapply(htmltools::HTML)

popups <- sprintf(
    "Year: %s<br/>Magnitude: %.1f",
    year(earthquakes$DateTime), earthquakes$Magnitude) %>% 
    lapply(htmltools::HTML)

magnitude_colors <- function(x) ifelse(x > 6.5, "red", "green")

In [ ]:
japan_density <- leaflet(prefectures) %>% 
    setView(lng = 136, lat = 35, zoom = 5) %>% 
    addProviderTiles("CartoDB.Positron") %>%
    addPolygons(color = "black",
                weight = 0.5,
                opacity = 0.8,
                fillColor = ~pal(stats$density),
                fillOpacity = 0.7,
                highlightOptions = highlightOptions(
                    weight = 1,
                    opacity = 1,
                    fillOpacity = 0.7,
                    bringToFront = TRUE,
                    sendToBack = TRUE
                ),
                label = labels,
                labelOptions = labelOptions(
                    style = list("font-weight" = "normal", padding = "3px 8px"),
                    textsize = "15px",
                    direction = "auto"
                )
    ) %>% 
    addLegend(pal = pal, 
              values = ~stats$density, 
              opacity = 0.7, 
              title = "Pop. Density",
              position = "bottomright"
    )

In [ ]:
japan_density %>% 
    addCircles(data = earthquakes,
               lng = ~Longitude, 
               lat = ~Latitude,
               fill = FALSE,
               radius = earthquakes$Magnitude ** 5.5,
               color = ~magnitude_colors(earthquakes$Magnitude),
               popup = popups
               )
